download data from: https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip

In [ ]:
import pandas as pd
import numpy as np

df_all = pd.read_sas(
    "../data/LLCP2023.XPT",
    format="xport"
)

In [ ]:
df_all.head()

In [ ]:
df = df_all.copy()

In [ ]:
df = df.loc[:,["CVDINFR4", "BPHIGH6", "TOLDHI3", "SMOKE100", "SMOKDAY2", "_BMI5", "EXERANY2", "ALCDAY4", "_AGE_G", "_SEX", "GENHLTH", "PHYSHLTH", "MENTHLTH", "POORHLTH", "ASTHMA3","CHCSCNC1", "CHCCOPD3", "CHCKDNY2", "DIABETE4", "CVDSTRK3", "DIFFWALK"]]
df.head()

In [ ]:
# rename columns
rename_map = {
    # Outcomes
    "CVDINFR4": "ever_heart_attack",

    # Clinical risk factors
    "BPHIGH6": "high_blood_pressure",
    "TOLDHI3": "high_cholesterol",
    "DIABETE4": "diabetes",
    "_BMI5": "bmi_x100",
    "DIFFWALK": "difficulty_walking",

    # Smoking
    "SMOKE100": "ever_smoked_100_cigs",
    "SMOKDAY2": "current_smoking_status",

    # Lifestyle
    "EXERANY2": "exercise_past_30_days",
    "ALCDAY4": "alcohol_days_per_month",

    # Demographics
    "_AGE_G": "age_group",
    "_SEX": "sex",

    # Self-rated health
    "GENHLTH": "general_health",
    "PHYSHLTH": "poor_physical_health_days",
    "MENTHLTH": "poor_mental_health_days",
    "POORHLTH": "activity_limited_health_days",

    # Comorbidities
    "ASTHMA3": "current_asthma",
    "CHCSCNC1": "skin_cancer_history",
    "CHCCOPD3": "copd_history",
    "CHCKDNY2": "kidney_disease_history",
    "CVDSTRK3": "stroke_history",
}
df = df.rename(columns=rename_map)
df.head()

In [ ]:
# walking difficulty
df["difficulty_walking"] = df["difficulty_walking"].replace({
    1: 1,
    2: 0,
    7: np.nan,
    9: np.nan
}).astype(float)


In [ ]:


def recode_binary(series):
    return (
        series
        .replace({
            1: 1,
            2: 0,
            7: np.nan,
            9: np.nan,
            77: np.nan,
            88: np.nan,
            99: np.nan
        })
        .astype("float")
    )

In [ ]:
binary_vars = [
    "ever_heart_attack",
    "ever_smoked_100_cigs",
    "exercise_past_30_days",
    "current_asthma",
    "skin_cancer_history",
    "copd_history",
    "kidney_disease_history",
    "stroke_history",
    "difficulty_walking",
    "high_cholesterol"
]

for v in binary_vars:
    df[v] = recode_binary(df[v])
df[binary_vars] = df[binary_vars].astype("Int64")

In [ ]:
# Smoking
df["current_smoker"] = np.nan

# If smokes now → smoker
df.loc[df["current_smoking_status"].isin([1, 2]), "current_smoker"] = 1

# If does not smoke now → non-smoker
df.loc[df["current_smoking_status"] == 3, "current_smoker"] = 0

# If never smoked ≥100 cigarettes → non-smoker
df.loc[df["ever_smoked_100_cigs"] == 0, "current_smoker"] = 0

# Explicit missing codes
df.loc[
    df["current_smoking_status"].isin([7, 9]) |
    df["ever_smoked_100_cigs"].isin([7, 9]),
    "current_smoker"
] = np.nan
df = df.drop("current_smoking_status", axis=1)

In [ ]:
# Days: health, mental health
def recode_days(s):
    return s.replace({
        88: 0,
        77: np.nan,
        99: np.nan
    })
df["poor_physical_health_days"] = recode_days(
    df["poor_physical_health_days"])
df["poor_mental_health_days"] = recode_days(
    df["poor_mental_health_days"])
df["activity_limited_health_days"] = recode_days(
    df["poor_physical_health_days"])

In [ ]:
# BMI
df["bmi"] = round(df["bmi_x100"]/100,2)
df = df.drop("bmi_x100", axis=1)

In [ ]:
# Sex at birth
df["sex"] = df["sex"].replace({
    1: "Male",
    2: "Female"
})

In [ ]:
# Age group
age_map = {
    1: "18-24",
    2: "25-29",
    3: "30-34",
    4: "35-39",
    5: "40-44",
    6: "45-49",
    7: "50-54",
    8: "55-59",
    9: "60-64",
    10: "65-69",
    11: "70-74",
    12: "75-79",
    13: "80+"
}

df["age_group"] = df["age_group"].map(age_map)

In [ ]:
# general health
genhlth_map = {
    1: "Excellent",
    2: "Very good",
    3: "Good",
    4: "Fair",
    5: "Poor",
    7: np.nan,
    9: np.nan,
}

df["general_health_label"] = df["general_health"].replace(genhlth_map)
df.loc[df["general_health_label"].isin([7, 9]), "general_health"] = np.nan
df = df.drop("general_health", axis=1)

In [ ]:
# blood pressure: suggestion for ML: use both during pregnancy and borderline as yes
df["high_blood_pressure_label"] = df["high_blood_pressure"].replace({
    1: "Yes",
    2: "Yes, during pregnancy",
    3: "No",
    4: "No, but Borderline",
    7: np.nan,
    9: np.nan
})
df = df.drop("high_blood_pressure", axis=1)

In [ ]:
# diabetes: suggestion for ML: use both during pregnancy and borderline as yes
df["diabetes_label"] = df["diabetes"].replace({
    1: "Yes",
    2: "Yes, during pregnancy",
    3: "No",
    4: "No, but Borderline",
    7: np.nan,
    9: np.nan
})
df = df.drop("diabetes", axis=1)

In [ ]:
# Alcoholdays/month
def alc_days_per_month(val):
    if val in [777, 999]:
        return np.nan
    elif val == 888:
        return 0
    elif 1 <= val <= 30:
        return val  # already monthly
    elif 101 <= val <= 199:
        # weekly drinking: subtract 100 to get days/week, multiply by 4
        return round((val - 100) * 4.3)
    else:
        return np.nan  # other unexpected codes

df["alcohol_days_month"] = df["alcohol_days_per_month"].apply(alc_days_per_month)
df = df.drop("alcohol_days_per_month", axis=1)

In [ ]:
df.sample(6)

In [ ]:
df.shape

In [ ]:
# drop rows with missing label
df = df.dropna(subset=["ever_heart_attack"])

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.to_csv("../data/brfss_2023_heart_risk_clean.csv", index=False)